# 01 · Data loading + checks
**Owner:** Senanayake · **Branch:** `data/01-loading-checks`

Read all 3 batches from the MATR/Severson `.mat` structs, apply the canonical cell removals + batch1↔batch2 continuation merge, and run the data-quality checks (missing / duplicate / invalid / cycle-order / ranges).


In [ ]:
# --- Colab setup (run first) ---
import os
if not os.path.exists('ev-battery-rul-prediction'):
    !git clone https://github.com/Chathumi-Senanayake/ev-battery-rul-prediction.git
%cd ev-battery-rul-prediction
!git pull -q
!pip install -q -r requirements.txt
import sys; sys.path.append('.')
from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/ev_battery_data'
os.makedirs(DATA_DIR, exist_ok=True)


### Download (run once; files persist on Drive)
Batch 3 URL is known. Fill Batch 1 & 2 from the [project page](https://data.matr.io/1/projects/5c48dd2bc625d700019f3204) (click each batch → right-click download → Copy link address).


In [ ]:
from src.data_loading import DOWNLOAD, MAT_FILENAMES
DOWNLOAD['batch1'] = 'PASTE_BATCH1_DOWNLOAD_URL_HERE'
DOWNLOAD['batch2'] = 'PASTE_BATCH2_DOWNLOAD_URL_HERE'
for b in ['batch1','batch2','batch3']:
    out = os.path.join(DATA_DIR, MAT_FILENAMES[b])
    if os.path.exists(out): print('have', MAT_FILENAMES[b]); continue
    if 'PASTE_' in DOWNLOAD[b]: print('skip', b, '- fill URL'); continue
    !wget -q -O '{out}' '{DOWNLOAD[b]}' && echo 'downloaded {b}'


### Load + merge


In [ ]:
import pickle
from src.data_loading import load_batch, merge_batches, batch_counts
b1 = load_batch(os.path.join(DATA_DIR, MAT_FILENAMES['batch1']), 'b1')
b2 = load_batch(os.path.join(DATA_DIR, MAT_FILENAMES['batch2']), 'b2')
b3 = load_batch(os.path.join(DATA_DIR, MAT_FILENAMES['batch3']), 'b3')
print('raw cells:', len(b1), len(b2), len(b3))
print('expected:', batch_counts(b1, b2, b3))
bat_dict = merge_batches(b1, b2, b3)
print('merged:', len(bat_dict), '(expect 124)')
pickle.dump(bat_dict, open(os.path.join(DATA_DIR,'bat_dict.pkl'),'wb'))


### Quality checks


In [ ]:
from src.preprocessing import build_summary_table, quality_report, print_report
df = build_summary_table(bat_dict)
print_report(quality_report(df))
df.head()
